Data loading

In [10]:
import os

import numpy as np
import pandas as pd
from PIL import Image


from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import (
    Conv2D, BatchNormalization, ReLU, Add, MaxPooling2D, Dense, Input
)
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout

# ==========================================
# Load augmented CSV
# ==========================================
name = "sasch"

df = pd.read_csv(
    fr"C:\Users\{name}\OneDrive\Desktop\iivp-2026-challenge\train_augmented.csv"
)

# ==========================================
# Load Images
# ==========================================

train_dir = fr"C:\Users\sasch\OneDrive\Desktop\iivp-2026-challenge\train_augmented"

X = []
y = []

for _, row in df.iterrows():

    img_path = os.path.join(
        train_dir,
        str(row["Category"]),
        str(row["Id"]) + ".png"
    )

    img = Image.open(img_path).convert("L")

    # normalize
    img = np.array(img) / 255.0

    X.append(img)
    y.append(row["Category"])

# ==========================================
# Convert to numpy and Reshape for The ResNet-Architecture ccn
# ==========================================

X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)

X = X.reshape(-1, 32, 32, 1)

print("CNN shape:", X.shape)

# ==========================================
# TRAIN / VALIDATION SPLIT
# ==========================================

X,y = shuffle(X, y,random_state=42)

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


Dataset shape: (34000, 32, 32)
CNN shape: (34000, 32, 32, 1)


Model creation

In [11]:

# ==========================================
# Residual Blocks for ResNet-Architecture
# ==========================================
def residual_block(x, filters):
    shortcut = x

    x = Conv2D(filters, (3,3), padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = Conv2D(filters, (3,3), padding="same")(x)
    x = BatchNormalization()(x)

    if shortcut.shape[-1] != filters:
        shortcut = Conv2D(filters, (1,1), padding="same")(shortcut)
        shortcut = BatchNormalization()(shortcut)

    x = Add()([x,shortcut])
    x = ReLU()(x)

    return x

# ==========================================
# Residual Blocks for ResNet-Architecture
# ==========================================
def build_model(drop_rat=0.3, base_filters=32):
    inputs = Input(shape=(32,32,1))

    x = Conv2D(base_filters,(3,3), padding="same")(inputs)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = residual_block(x, base_filters)
    x = residual_block(x, base_filters)

    x = MaxPooling2D()(x)

    x = Conv2D(base_filters*2,(3,3), padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = residual_block(x, base_filters*2)
    x = residual_block(x, base_filters*2)

    x = MaxPooling2D()(x)

    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation="relu")(x)
    x = Dropout(drop_rat)(x)

    outputs = Dense(10, activation="softmax")(x)

    model = Model(inputs, outputs)

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"])

    return model

Model Training

In [12]:
# ==========================================
# Data Augmentation
# ==========================================
datagen = ImageDataGenerator(
    rotation_range=8,
    width_shift_range=0.08,
    height_shift_range=0.08,
    zoom_range=0.08
)

# ==========================================
# Adding Noise (not efficient so removed)
# ==========================================

# def add_noise(images):
#     noise = np.random.normal(0,0.02, images.shape)
#     return np.clip(images + noise, 0., 1.)
#
# X_train = add_noise(X_train)

# ==========================================
# Callbacks (helps stopping early if the learning flattens to much
# ==========================================
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=3,
    min_lr=0.0001,
    verbose=1
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=6,
    restore_best_weights=True,
    verbose=1
)

# ==========================================
# Train
# ==========================================

models = []
ensemble_size = 3

for i in range(ensemble_size):
    print(f"Training model {i+1}/{ensemble_size}")

    np.random.seed(42+i)
    tf.random.set_seed(42+i)

    drop_rate = 0.2+0.05*i

    model = build_model(drop_rat=drop_rate, base_filters=32)

    train_generator = datagen.flow(
        X_train,
        y_train,
        batch_size=32
    )

    history = model.fit(
    train_generator,
    epochs=30,
    validation_data=(X_val, y_val),
    callbacks = [lr_scheduler, early_stopping],
    verbose=1
    )

    model.save(f"model_{i}.h5")
    models.append(model)

# ==========================================
# Validation Accuracy
# ==========================================
print("\nEvaluating Ensemble")

val_preds = np.mean([m.predict(X_val) for m in models], axis=0)
val_labels = np.argmax(val_preds, axis=1)

val_acc = np.mean(val_labels == y_val)
print("Ensemble Validation Accuracy:", val_acc)

Training model 1/3
Epoch 1/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 126s 143ms/step - accuracy: 0.9137 - loss: 0.2783 - val_accuracy: 0.9125 - val_loss: 0.2804 - learning_rate: 0.0010
Epoch 2/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 113s 132ms/step - accuracy: 0.9842 - loss: 0.0545 - val_accuracy: 0.9263 - val_loss: 0.2367 - learning_rate: 0.0010
Epoch 3/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 111s 130ms/step - accuracy: 0.9877 - loss: 0.0413 - val_accuracy: 0.9899 - val_loss: 0.0340 - learning_rate: 0.0010
Epoch 4/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 111s 130ms/step - accuracy: 0.9907 - loss: 0.0317 - val_accuracy: 0.9924 - val_loss: 0.0226 - learning_rate: 0.0010
Epoch 5/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 113s 133ms/step - accuracy: 0.9908 - loss: 0.0284 - val_accuracy: 0.9669 - val_loss: 0.1032 - learning_rate: 0.0010
Epoch 6/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 111s 131ms/step - accuracy: 0.9915 - loss: 0.0279 - val_accuracy: 0.9866 - val_loss: 0.0443 - learning_rate: 0.0010
Epoch 7/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 2451s

Training model 2/3
Epoch 1/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 114s 129ms/step - accuracy: 0.9133 - loss: 0.2832 - val_accuracy: 0.8984 - val_loss: 0.3028 - learning_rate: 0.0010
Epoch 2/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 114s 134ms/step - accuracy: 0.9841 - loss: 0.0549 - val_accuracy: 0.9471 - val_loss: 0.1627 - learning_rate: 0.0010
Epoch 3/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step - accuracy: 0.9870 - loss: 0.0438
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0003000000142492354.
850/850 ━━━━━━━━━━━━━━━━━━━━ 116s 136ms/step - accuracy: 0.9889 - loss: 0.0379 - val_accuracy: 0.9954 - val_loss: 0.0173 - learning_rate: 0.0010
Epoch 4/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 111s 130ms/step - accuracy: 0.9962 - loss: 0.0136 - val_accuracy: 0.9963 - val_loss: 0.0143 - learning_rate: 3.0000e-04
Epoch 5/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 108s 127ms/step - accuracy: 0.9960 - loss: 0.0147 - val_accuracy: 0.9988 - val_loss: 0.0047 - learning_rate: 3.0000e-04
Epoch 6/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 

Training model 3/3
Epoch 1/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 124s 140ms/step - accuracy: 0.9110 - loss: 0.2929 - val_accuracy: 0.9291 - val_loss: 0.1914 - learning_rate: 0.0010
Epoch 2/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 114s 134ms/step - accuracy: 0.9818 - loss: 0.0622 - val_accuracy: 0.9791 - val_loss: 0.0669 - learning_rate: 0.0010
Epoch 3/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step - accuracy: 0.9866 - loss: 0.0457
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0003000000142492354.
850/850 ━━━━━━━━━━━━━━━━━━━━ 111s 130ms/step - accuracy: 0.9876 - loss: 0.0413 - val_accuracy: 0.9729 - val_loss: 0.0878 - learning_rate: 0.0010
Epoch 4/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 110s 129ms/step - accuracy: 0.9955 - loss: 0.0165 - val_accuracy: 0.9972 - val_loss: 0.0081 - learning_rate: 3.0000e-04
Epoch 5/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 107s 126ms/step - accuracy: 0.9967 - loss: 0.0117 - val_accuracy: 0.9993 - val_loss: 0.0024 - learning_rate: 3.0000e-04
Epoch 6/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 


Evaluating Ensemble
213/213 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step
213/213 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step
213/213 ━━━━━━━━━━━━━━━━━━━━ 7s 33ms/step
Ensemble Validation Accuracy: 0.9967647058823529


Inference

In [14]:
models = [tf.keras.models.load_model(f"model_{i}.h5") for i in range(ensemble_size)]

# ==========================================
# Load Test set
# ==========================================
X_test = []
names = []

test_dir = rf"C:\Users\{name}\OneDrive\Desktop\iivp-2026-challenge\test\test"

for file in sorted(os.listdir(test_dir)):
    img = Image.open(os.path.join(test_dir, file)).convert("L")
    img = np.array(img) / 255.0
    X_test.append(img)
    names.append(file)

X_test = np.array(X_test).reshape(-1, 32, 32, 1)

# ==========================================
# Prediction with TTA (not efficient, so removed)
# ==========================================

# final_preds = []
#
# for model in models:
#     tta_preds = []
#
#     for _ in range(5):
#         aug_iter = datagen.flow(X_test, shuffle=False, batch_size=len(X_test))
#         X_aug = next(aug_iter)
#         tta_preds.append(model.predict(X_aug))
#     final_preds.append(np.mean(tta_preds, axis=0))
#
# preds = np.mean(final_preds, axis=0)

# ==========================================
# Prediction
# ==========================================
preds = np.mean([m.predict(X_test) for m in models], axis=0)


# ==========================================
# adding Probability sharpening (not efficient, so removed)
# ==========================================
# preds = preds ** 1.2
# preds = preds / np.sum(preds, axis=1, keepdims=True)


# ==========================================
# Submission CSV
# ==========================================
pred_labels = np.argmax(preds, axis=1)

submission = pd.DataFrame({
    "Id": [x.replace(".png","") for x in names],
    "Category": pred_labels
})

submission.to_csv("submission_ResNetWithoutTTAandProb.csv", index=False)
print("submission_cnn.csv saved")


94/94 ━━━━━━━━━━━━━━━━━━━━ 5s 50ms/step
94/94 ━━━━━━━━━━━━━━━━━━━━ 5s 51ms/step
94/94 ━━━━━━━━━━━━━━━━━━━━ 6s 55ms/step
submission_cnn.csv saved
